# Module 4: Review Accounting Scope and Build a Pilot Research Sample

本模块读取 Module 3 已保存的结果，不重新下载数据，也不修改 Module 3 的输出。
教学说明用中文，代码、注释、列名和运行提示用英文。可接在原 Notebook 后面，也可单独运行；项目目录需一致。

我们将建立三份不同用途的表：全部年度的审核工作表、金额齐全且通过基础检查的试点年度表，以及能计算连续年度变化的试点特征表。
三家公司仍是流程试点，不是最终 30–50 家研究样本。

本模块完成**基础金额筛查和已知口径差异记录**，不声称所有历史年报、标签、终止经营和财务重列均已审核。
保留 pending 状态，不把三个公司的毛利率水平直接进行优劣排名。
本模块计算当期及同比指标，下一模块再建立下一期结果与披露时间规则；当前表不直接用于投资回测。


## Cell 1: Set Up the Project

沿用前面相同的目录结构。`Path.cwd()` 是当前 Jupyter 的工作目录，不一定是 Notebook 文件所在目录。
你上一版的项目目录输出为 `[project root]`；继续在同一环境运行即可。若工作目录改变，应将 PROJECT_ROOT 改为实际项目目录。
已有 pandas，无需再次安装。`PROCESSED_DIR` 指向已整理数据目录。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in ["notebooks", "guides", "outputs"]:
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_DIR = PROJECT_ROOT / "work" / "data" / "processed"
print("Project directory:", PROJECT_ROOT)


## Cell 2: Read the Saved Module 3 Results

`panel` 读取年度主表；`sources` 读取逐字段来源；`module3_metadata` 保留原处理规则和快照时间。
`parse_dates` 把 CSV 中的日期字符串恢复成日期对象，后面才能计算日期差。
`.sort_values()` 保证同一家公司内部按年度先后排列；`.reset_index(drop=True)` 只重新编号，不删除记录。
`sources` 中缺失的 tag 可表示计算值，不等于金额缺失。查看状态分布，用于确认修复结果已保存。


In [ ]:
panel = pd.read_csv(
    PROCESSED_DIR / "pilot_annual_panel.csv",
    parse_dates=["period_start", "period_end", "filed"],
)
sources = pd.read_csv(PROCESSED_DIR / "pilot_annual_field_sources.csv")
module3_metadata = json.loads(
    (PROCESSED_DIR / "pilot_annual_panel_metadata.json").read_text(encoding="utf-8")
)
panel = panel.sort_values(["ticker", "period_end"]).reset_index(drop=True)
metrics = ["revenue", "cost_of_sales", "gross_profit", "inventory", "total_assets", "operating_cash_flow"]
print("Annual rows loaded:", len(panel))
display(sources["status"].value_counts().to_frame("Field Count"))


## Cell 3: Record Company Scope Notes

把已经核实的重要差异写成英文记录，便于最后报告引用。
这些是口径说明，不是程序对全部历史年份自动完成的审计。

- AEO：销售成本包括部分采购、占用和仓储费用；2013 年提交年报将 77kids 列为终止经营，并重列比较年度。因此原始不同年报之间的增长可能混入业务范围变化。
- ANF：销售成本排除折旧摊销，其成本费用范围与其他公司不完全一致。
- URBN：公司合并数据包括零售、批发和订阅；部分年度毛利还扣除了单独列示的门店减值及租赁放弃费用。

近期披露只能支持近期口径。历史口径仍需逐年核对。AEO 终止经营信息来自对应的历史年报。
来源：[AEO 历史年报](https://www.sec.gov/Archives/edgar/data/919012/000119312513102293/d468081d10k.htm)、[AEO 成本口径](https://www.sec.gov/Archives/edgar/data/919012/000119312515087519/d848537d10k.htm)、[ANF 年报](https://www.sec.gov/Archives/edgar/data/1018840/000101884026000012/anf-20260131.htm)、[URBN 年报](https://www.sec.gov/Archives/edgar/data/912615/000119312526138040/urbn_fy26_10-k_-_ars.pdf)。


In [ ]:
scope_notes = pd.DataFrame([
    {"ticker": "AEO", "cost_scope": "Includes certain buying, occupancy, and warehousing expenses.",
     "business_scope": "Historical 77kids discontinued-operation reclassification requires review.",
     "source_url": "https://www.sec.gov/Archives/edgar/data/919012/000119312513102293/d468081d10k.htm"},
    {"ticker": "ANF", "cost_scope": "Excludes depreciation and amortization; expense classifications differ across retailers.",
     "business_scope": "Consolidated company data; historical scope review is pending.",
     "source_url": "https://www.sec.gov/Archives/edgar/data/1018840/000101884026000012/anf-20260131.htm"},
    {"ticker": "URBN", "cost_scope": "Reported gross profit can also deduct separately presented store charges.",
     "business_scope": "Includes retail, wholesale, and subscription operations.",
     "source_url": "https://www.sec.gov/Archives/edgar/data/912615/000119312526138040/urbn_fy26_10-k_-_ars.pdf"},
])
scope_notes["historical_scope_review"] = "pending"
display(scope_notes)


## Cell 4: Flag Missing Amounts and Accounting Differences

`.copy()` 建立独立审核表，保持原始 `panel` 不变。
`notna().all(axis=1)` 检查每行的六个金额是否齐全；`axis=1` 表示逐行。
基础检查要求收入、资产为正，存货和成本非负。负毛利与负经营现金流可以真实存在，因此不因其为负而删除。
`row_usable` 仅表示符合本模块的金额筛查规则，不等于完成会计口径审核。

`gross_profit_bridge_gap = revenue - cost_of_sales - gross_profit` 展示毛利桥接差额。绝对差额超过 1 美元的行供核对。
URBN 可以因单独扣除门店费用而有差额，不能据此直接判定数据错误或自动修正。
用收入与毛利计算出来的 AEO 成本，差额自然为零；这个零不是一次独立验证。

`known_scope_break` 标记 AEO 年度终点 2013-02-02 的跨年口径变化。保留当年的金额和比率，但暂时排除该跨年增长比较。
该标记只覆盖目前已经核实的一处变化，不代表其他年份已完全可比。


In [ ]:
audit = panel.copy()
audit["amounts_available"] = audit[metrics].notna().all(axis=1)
audit["basic_values_ok"] = (
    (audit["revenue"] > 0) & (audit["total_assets"] > 0)
    & (audit["inventory"] >= 0) & (audit["cost_of_sales"] >= 0)
)
audit["row_usable"] = audit["amounts_available"] & audit["basic_values_ok"]
audit["gross_profit_bridge_gap"] = audit["revenue"] - audit["cost_of_sales"] - audit["gross_profit"]
audit["known_scope_break"] = (
    (audit["ticker"] == "AEO") & (audit["period_end"] == pd.Timestamp("2013-02-02"))
)
display(audit.groupby("ticker")[["amounts_available", "row_usable"]].sum())
display(audit.loc[~audit["row_usable"], ["ticker", "period_end"] + metrics])
display(audit.loc[
    audit["gross_profit_bridge_gap"].abs() > 1,
    ["ticker", "period_end", "gross_profit_bridge_gap"],
])


## Cell 5: Calculate Current-Period Ratios

先计算三个当期比率：毛利率、经营现金流收入比、存货资产比。
比率用小数保存，0.10 表示 10%。经营现金流收入比不等于利润率，也不是“净利润转化为现金”的比例。
最后一行把不符合基础金额筛查的行的新增比率留空；不改变任何原始金额。

毛利率优先使用已保存的毛利值，包括已标记的 ANF 计算值。不能对所有公司统一改成收入减成本。
存货资产比会受资产构成影响，例如租赁会计政策变化；现在先用于描述，不用它直接判断公司优劣。


In [ ]:
audit["gross_margin"] = audit["gross_profit"] / audit["revenue"]
audit["ocf_margin"] = audit["operating_cash_flow"] / audit["revenue"]
audit["inventory_to_assets"] = audit["inventory"] / audit["total_assets"]
ratio_columns = ["gross_margin", "ocf_margin", "inventory_to_assets"]
audit.loc[~audit["row_usable"], ratio_columns] = float("nan")
display(audit[["ticker", "period_end"] + ratio_columns].tail(6))


## Cell 6: Calculate Consecutive Annual Changes

`.groupby("ticker")` 将公司分开，`.shift(1)` 取同一公司上一行，避免把 ANF 的上一年取成 AEO。
之前必须先排序；“上一行”只有日期间隔合理时才当作上一年度。

`growth_usable` 要求当前与上一年度都通过金额筛查，年度终点间隔为 350–380 天，增长率的上一年分母为正，且没有当前已知的口径断点。
先在完整审核表上计算上一期，再筛选最终样本，避免删除缺失年份后把两年前误当成上一年。
`.eq(True)` 使没有上一行的第一年得到 False。

收入与存货增长率用“本年 / 上年 - 1”。`inventory_sales_gap` 是存货增长率减收入增长率：0.05 表示存货增长高出收入增长 **5 个百分点**，不是 5% 的相对增幅。
毛利率和经营现金流收入比的同比变化用直接相减；例如 0.40 到 0.37 得到 -0.03，即下降 3 个百分点。
`period_length_changed` 标记年度长度相差超过 3 天的比较，主要用于提醒 52/53 周差异；不自动调整收入或删除这些比较。

正的库存增长差只能说明值得进一步检查，不能直接推出库存积压、利润下降或卖出结论。


In [ ]:
for column in ["revenue", "inventory", "gross_margin", "ocf_margin", "period_end", "period_days"]:
    audit["previous_" + column] = audit.groupby("ticker")[column].shift(1)
previous_row_usable = audit.groupby("ticker")["row_usable"].shift(1).eq(True)
year_gap = (audit["period_end"] - audit["previous_period_end"]).dt.days
audit["growth_usable"] = (
    audit["row_usable"] & previous_row_usable & year_gap.between(350, 380)
    & (audit["previous_revenue"] > 0) & (audit["previous_inventory"] > 0)
    & ~audit["known_scope_break"]
)
audit["period_length_changed"] = (
    (audit["period_days"] - audit["previous_period_days"]).abs() > 3
)
audit["revenue_growth"] = audit["revenue"] / audit["previous_revenue"] - 1
audit["inventory_growth"] = audit["inventory"] / audit["previous_inventory"] - 1
audit["inventory_sales_gap"] = audit["inventory_growth"] - audit["revenue_growth"]
audit["gross_margin_change"] = audit["gross_margin"] - audit["previous_gross_margin"]
audit["ocf_margin_change"] = audit["ocf_margin"] - audit["previous_ocf_margin"]
change_columns = ["revenue_growth", "inventory_growth", "inventory_sales_gap", "gross_margin_change", "ocf_margin_change"]
audit.loc[~audit["growth_usable"], change_columns] = float("nan")
display(audit[["ticker", "period_end", "growth_usable", "period_length_changed"] + change_columns].tail(6))


## Cell 7: Build the Pilot Samples

`research_panel` 保留六项金额齐全且通过基础筛查的年度；它适合当期财务描述。
`inventory_features` 再要求连续年度增长可以计算，并排除已知口径断点；它适合后续库存信号研究。

你目前的缓存预计得到：research_panel 51 行；inventory_features 47 行（AEO 15、ANF 16、URBN 16）。
2009 年缺少存货与资产；2010 年虽有完整金额，但缺少合格的上一年基期，因此不能进入增长特征表。
其余 48 个连续年度比较中，AEO 2013-02-02 的比较再排除一个，得到 47 个。

这 47 行并不是已经有“下一年结果”的预测样本。下一模块还要配对下一年度结果；最新年度可能没有对应结果。
样本标签使用实际 period_end，不把它误称为三家公司完全一致的财年年份。


In [ ]:
research_panel = audit.loc[audit["row_usable"]].copy()
inventory_features = audit.loc[audit["growth_usable"]].copy()
print("Complete pilot annual rows:", len(research_panel))
print("Eligible pilot growth rows:", len(inventory_features))
display(inventory_features.groupby("ticker").size().to_frame("Growth Rows"))
display(inventory_features.loc[
    inventory_features["ticker"] == "AEO",
    ["ticker", "period_end", "period_length_changed"] + change_columns,
].tail(5))


## Cell 8: Save the Pilot Samples and Review Notes

普通字典将文件名与 DataFrame 对应，循环逐个保存。再次运行会覆盖 Module 4 的同名输出，但不覆盖 Module 3 文件。
metadata 记录原数据处理说明、本模块筛选规则、样本数量和仍未完成的审核。
审计工作表保留 54 行；来源金额和公式仍可回查 Module 3 的 field_sources 文件。

`accounting_scope_verified` 和 `prediction_sample_ready` 都保留 False：历史业务范围、重列、口径变化和预测披露时间规则尚需进一步审核。


In [ ]:
outputs = {
    "pilot_annual_audit.csv": audit,
    "pilot_company_scope_notes.csv": scope_notes,
    "pilot_research_panel.csv": research_panel,
    "pilot_inventory_features.csv": inventory_features,
}
for filename, data in outputs.items():
    data.to_csv(PROCESSED_DIR / filename, index=False, date_format="%Y-%m-%d")
    print("Saved:", filename)
metadata = {
    "processed_at_utc": datetime.now(timezone.utc).isoformat(),
    "module3_metadata": module3_metadata,
    "unit": "USD; ratios and changes stored as decimals",
    "complete_annual_rows": len(research_panel),
    "eligible_growth_rows": len(inventory_features),
    "known_scope_break": "Exclude AEO growth comparison ending 2013-02-02 due to discontinued-operation reclassification.",
    "growth_rule": "Consecutive complete annual rows, 350-380 day end-date gap, positive prior revenue and inventory, excluding known scope breaks.",
    "period_length_policy": "Flag material length changes; do not normalize revenue automatically.",
    "accounting_scope_verified": False,
    "prediction_sample_ready": False,
}
(PROCESSED_DIR / "pilot_module4_metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)
print("Saved: pilot_module4_metadata.json")


## Completion and Next Step

本模块先把技术上可计算的试点表与已知口径差异记录下来。不要因此宣称 47 条记录均已完成历史可比性审核。
下一步将建立“本期库存增长差 → 下一年度毛利率与经营现金流表现”的配对样本，核对 filed 时间，并逐步扩大公司数量。
是否存在预测关系需要数据分析验证；三家公司只能用于验证处理流程与展示个案。
